# P2 · Act: dari data sensor ke model

**Kasusnya:** kalian baru kerja di pabrik komponen otomotif. Plant manager datang dan bilang:

> *"Kasih tau saya mesin mana yang kemungkinan rusak dalam 24 jam ke depan, biar bisa diservis duluan."*

Di akhir notebook ini kalian punya model yang bisa menjawab itu, disimpan jadi tiga file:
`model.joblib`, `metadata.json`, dan `src/features.py`. Tiga file ini dibawa ke P3.

**Cara pakai**
1. Klik **File → Save a copy in Drive** dulu. Kalau nggak, kerjaan kalian nggak kesimpen.
2. Jalankan sel dari atas ke bawah pakai **Shift + Enter**.
3. Ada **8 TODO** yang harus kalian isi. Di bawah tiap TODO ada sel **✅ Cek**. Kalau sel Cek jalan tanpa
   error, jawaban kalian benar. Kalau error, baca pesannya, biasanya sudah ada petunjuknya.
4. Mentok? Tempel pesan error lengkapnya di chat kelas atau di QnA.

> ⚠️ Jangan jalankan sel TODO sebelum diisi. Isian `...` bakal menimpa variabelnya.
> Kalau terlanjur, pilih **Runtime → Run before** di sel itu supaya semua sel di atasnya jalan ulang.

| Bagian | Isi | Kalian ngoding? |
|---|---|---|
| 1–2 | Setup dan kenalan sama data | tinggal jalankan |
| 3 | Bersih-bersih data | TODO 1–3 |
| 4 | Fitur dari rumus fisika | TODO 4 |
| 5 | Bagi data jadi train, validation, test | TODO 5 |
| 6 | Pipeline dan model pertama (Random Forest) | tinggal jalankan |
| 7 | XGBoost dan fitur yang paling berpengaruh | TODO 6 |
| 8 | Threshold berdasarkan biaya, lalu buka test set | TODO 7 |
| 9 | Simpan model | TODO 8 |
| 10 | Coba pakai model dari file | tinggal jalankan |

## 1 · Setup
Sel ini mengunduh dataset dan menyiapkan folder. Sudah jadi, tinggal jalankan.

In [ ]:
import json, os, sys, time, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/nurhikam/mkb-ml-p2/main/data/mesin_cnc.csv"
if not os.path.exists("mesin_cnc.csv"):
    urllib.request.urlretrieve(DATA_URL, "mesin_cnc.csv")

os.makedirs("src", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)
open("src/__init__.py", "a").close()
if "" not in sys.path:
    sys.path.insert(0, "")

RANDOM_STATE = 42
df = pd.read_csv("mesin_cnc.csv")
print(f"{len(df):,} baris, {df.shape[1]} kolom")
df.head()

## 2 · Kenalan sama data

Jangan langsung training. Lihat dulu datanya kayak apa.

Ada sembilan kolom input dan satu target:

| Kolom | Artinya |
|---|---|
| `tipe_produk` | L / M / H, kualitas produk yang sedang dikerjakan |
| `suhu_udara_K`, `suhu_proses_K` | suhu udara sekitar dan suhu proses, dalam Kelvin |
| `kecepatan_rpm`, `torsi_Nm` | kecepatan putar spindle dan torsinya |
| `keausan_tool_menit` | sudah berapa menit tool (mata potong) dipakai |
| `getaran_mm_s` | getaran mesin |
| `jam_sejak_servis` | berapa jam sejak servis terakhir |
| `shift_operator` | pagi / siang / malam |
| `gagal_24jam` | **target**: 1 kalau mesin rusak dalam 24 jam ke depan, 0 kalau aman |

Datanya sintetis (data pabrik asli biasanya rahasia), tapi sengaja dibuat kotor seperti data lapangan.
Sambil lihat output di bawah, coba cari: mana nilai yang **pasti salah**, dan mana yang cuma **aneh tapi mungkin**?

### Tipe data
Langkah pertama: cek tipe data tiap kolom. `float64` dan `int64` itu angka. `object` itu teks.
Kolom teks nanti perlu diubah jadi angka dulu sebelum masuk model (di bagian 6).

In [ ]:
df.info()

Kalau mau lihat datanya kayak di Excel (bisa di-sort dan di-filter), jalankan sel di bawah.
Colab akan minta izin akses Google Drive karena datanya dibuka sebagai Google Sheet. Sel ini **opsional**.

In [ ]:
# Opsional: buka data sebagai tabel interaktif. Coba sort kolom torsi_Nm dari yang terkecil.
try:
    from google.colab import sheets
    sheet = sheets.InteractiveSheet(df=df)
except ImportError:
    print("Bukan di Colab, lewati sel ini.")

### Ringkasan angka
`describe()` menampilkan nilai minimum, maksimum, dan rata-rata tiap kolom angka.
Perhatikan baris **min**: ada yang nggak masuk akal?

In [ ]:
df.describe().round(2)

In [ ]:
print("Nilai kosong per kolom:")
print(df.isna().sum()[df.isna().sum() > 0], "\n")
print("Isi kolom shift_operator:")
print(df["shift_operator"].value_counts(), "\n")
print("Baris duplikat:", df.duplicated().sum())

### Berapa banyak mesin yang rusak?
Ini angka paling penting di bagian ini. Ingat-ingat, nanti dipakai lagi.

In [ ]:
rasio = df["gagal_24jam"].mean()
print(f"Mesin yang gagal: {rasio:.1%}")
print(f"Model yang SELALU bilang 'aman' sudah dapat akurasi {1 - rasio:.1%}, tanpa belajar apa pun.")

### Mesin seperti apa yang sering rusak?
Grafik di bawah menunjukkan **persentase mesin rusak** di tiap kelompok. Batang yang tinggi berarti
kelompok itu lebih sering rusak. Garis putus-putus adalah rata-rata semua mesin.

In [ ]:
lihat = df.dropna(subset=["keausan_tool_menit", "torsi_Nm"]).copy()
lihat["kelompok_keausan"] = pd.cut(lihat["keausan_tool_menit"], [0, 50, 100, 150, 200, 300],
                                   include_lowest=True)
lihat["kelompok_torsi"] = pd.cut(lihat["torsi_Nm"], [0, 20, 30, 40, 50, 60, 100], include_lowest=True)
lihat["shift"] = lihat["shift_operator"].str.strip().str.lower()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, kolom, judul in [(axes[0], "kelompok_keausan", "keausan tool (menit)"),
                         (axes[1], "kelompok_torsi", "torsi (Nm)"),
                         (axes[2], "shift", "shift operator")]:
    (lihat.groupby(kolom, observed=True)["gagal_24jam"].mean() * 100).plot.bar(ax=ax, color="tab:red", alpha=0.7)
    ax.axhline(rasio * 100, color="gray", linestyle="--")
    ax.set_title(f"% rusak per {judul}")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("% mesin rusak")
plt.tight_layout()
plt.show()

Apa yang kelihatan?

- **Keausan tool**: aman-aman saja sampai sekitar 200 menit, setelah itu persentase rusaknya melonjak.
- **Torsi**: bentuknya kayak huruf U. Torsi yang terlalu rendah *dan* terlalu tinggi sama-sama lebih sering rusak.
  Polanya nggak lurus, dan model berbasis pohon (yang kita pakai nanti) bisa menangkap pola seperti ini.
- **Shift operator**: shift malam sedikit lebih tinggi, tapi bedanya kecil.

Ingat pola keausan dan torsi ini, nanti dipakai di bagian 4.

## 3 · Bersih-bersih: yang salah dikosongkan, yang aneh dibiarkan

- **Salah** artinya mustahil secara fisika. Torsi minus atau suhu proses 0 Kelvin itu nggak mungkin,
  berarti sensornya error.
- **Aneh** artinya jarang tapi bisa terjadi, misalnya getaran tinggi. Justru mesin yang getarannya tinggi
  mungkin yang mau rusak. **Jangan dibuang.**

Nilai yang salah diganti jadi kosong (`NaN`), barisnya **jangan dihapus**. Di baris itu biasanya cuma satu
sensor yang error, kolom lainnya masih benar. Lagi pula, nanti waktu model dipakai beneran, sensor rusak juga
bakal kejadian, jadi model harus terbiasa ketemu nilai kosong. Nilai kosongnya diisi nanti di bagian 6.

### TODO 1 · buang baris duplikat

In [ ]:
# TODO 1: buang baris yang persis sama. Petunjuk: pandas punya method drop_duplicates()
df = ...

In [ ]:
# ✅ Cek TODO 1
assert isinstance(df, pd.DataFrame), "df harus tetap DataFrame. Sudah isi TODO 1?"
assert df.duplicated().sum() == 0, "Masih ada baris duplikat."
assert len(df) == 10_000, f"Harusnya 10,000 baris unik, sekarang {len(df):,}."
print("✅ TODO 1 beres:", f"{len(df):,} baris unik")

### TODO 2 · nilai mustahil jadi NaN
Torsi di bawah 0 dan suhu proses 0 Kelvin (atau kurang) itu sensor error.

In [ ]:
# TODO 2: ubah nilai yang mustahil jadi NaN (jangan hapus barisnya)
# Petunjuk: df.loc[kondisi, "nama_kolom"] = np.nan
...
...

In [ ]:
# ✅ Cek TODO 2
assert (df["torsi_Nm"] < 0).sum() == 0, "Masih ada torsi negatif."
assert (df["suhu_proses_K"] <= 0).sum() == 0, "Masih ada suhu proses 0 K."
assert len(df) == 10_000, "Jumlah baris berubah. Ubah jadi NaN, jangan hapus barisnya."
assert df["getaran_mm_s"].max() > 6, "Getaran tinggi ikut terhapus? Itu aneh tapi benar, simpan."
print("✅ TODO 2 beres:", df["torsi_Nm"].isna().sum(), "torsi dan",
      df["suhu_proses_K"].isna().sum(), "suhu proses jadi NaN")

### TODO 3 · rapikan kolom shift
Ada `"Malam "`, `"MALAM"`, `" siang"`. Buat komputer itu semua beda. Harusnya cuma tiga: `pagi`, `siang`, `malam`.

In [ ]:
# TODO 3: buang spasi di ujung dan jadikan huruf kecil semua
# Petunjuk: kolom teks punya .str.strip() dan .str.lower()
df["shift_operator"] = ...

In [ ]:
# ✅ Cek TODO 3
assert set(df["shift_operator"].unique()) == {"pagi", "siang", "malam"}, \
    f"Masih ada kategori aneh: {sorted(df['shift_operator'].unique())}"
print("✅ TODO 3 beres:", df["shift_operator"].value_counts().to_dict())

## 4 · Fitur dari rumus fisika

Model cuma lihat angka per kolom. Dia nggak tahu bahwa torsi dan rpm kalau digabung jadi daya.
Kita tahu rumusnya, jadi kita kasih langsung sebagai kolom baru:

| Fitur | Rumus | Artinya |
|---|---|---|
| `daya_W` | torsi × 2π × rpm / 60 | daya mekanik, dalam watt |
| `delta_suhu_K` | suhu proses − suhu udara | kalau selisihnya kecil, panas susah dibuang |
| `strain_tool` | keausan × torsi | tool yang sudah aus dan sedang menahan beban berat |

Ingat grafik di bagian 2: tool yang sudah aus dan torsi yang tinggi sama-sama bikin mesin lebih sering rusak.
`strain_tool` menggabungkan keduanya jadi satu angka.

Fungsinya **ditulis ke file `src/features.py`**, bukan cuma di sel notebook. Rumus ini nanti harus dipakai lagi
waktu modelnya dipakai, dan hasilnya harus sama persis. Kalau diketik ulang di tempat lain lalu beda sedikit
(misalnya lupa dibagi 60), model tetap jalan tanpa error tapi tebakannya ngaco.

### TODO 4 · lengkapi `tambah_fitur`
Isi tiga baris di sel di bawah, lalu jalankan. Sel ini menulis file `src/features.py`.

In [ ]:
%%writefile src/features.py
"""Fitur fisika. Dipakai saat training dan saat model dipakai, jadi jangan diubah di satu tempat saja."""
import numpy as np
import pandas as pd


def tambah_fitur(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # TODO 4: lengkapi tiga fitur di bawah (lihat tabel rumus di atas)
    df["daya_W"] = ...
    df["delta_suhu_K"] = ...
    df["strain_tool"] = ...
    return df

In [ ]:
# ✅ Cek TODO 4
import importlib
import src.features
importlib.reload(src.features)
from src.features import tambah_fitur

contoh = pd.DataFrame([{"torsi_Nm": 40.0, "kecepatan_rpm": 1500, "suhu_proses_K": 310.0,
                        "suhu_udara_K": 300.0, "keausan_tool_menit": 100}])
hasil = tambah_fitur(contoh).iloc[0]
assert abs(hasil["daya_W"] - 6283.19) < 1, f"daya_W salah: {hasil['daya_W']}"
assert abs(hasil["delta_suhu_K"] - 10.0) < 1e-6, f"delta_suhu_K salah: {hasil['delta_suhu_K']}"
assert abs(hasil["strain_tool"] - 4000.0) < 1e-6, f"strain_tool salah: {hasil['strain_tool']}"
assert "daya_W" not in contoh, "Jangan ubah DataFrame aslinya; fungsi harus pakai df.copy()."
print("✅ TODO 4 beres:", hasil[["daya_W", "delta_suhu_K", "strain_tool"]].round(2).to_dict())

## 5 · Bagi data: 60 / 20 / 20

Gampangnya pakai analogi ujian:

- **train** (6,000 baris) = latihan soal. Model belajar dari sini.
- **validation** (2,000) = try out. Dipakai buat milih model dan threshold.
- **test** 🔒 (2,000) = ujian beneran. Dikunci, **baru dibuka sekali di akhir**.

Kalau soal ujian dipakai buat latihan, nilainya pasti bagus, tapi nggak menunjukkan kemampuan sebenarnya.
Sama dengan model: kalau test set diintip waktu tuning, skornya jadi nggak bisa dipercaya.

`stratify` menjaga persentase mesin rusak (sekitar 5,5%) tetap sama di ketiga bagian.

### TODO 5
`train_test_split` cuma bisa memotong data jadi **dua**. Jadi kita potong dua kali:

1. 100% data → **train 60%** + **sisa 40%**. Yang 40% ini disimpan sementara di `X_sisa` dan `y_sisa`,
   artinya data yang belum kepakai.
2. `X_sisa` (40%) → **validation 20%** + **test 20%**, dibagi sama besar.

`X` itu kolom-kolom input, `y` itu jawabannya (kolom `gagal_24jam`). Keduanya selalu dipotong bareng.

In [ ]:
from sklearn.model_selection import train_test_split

TARGET = "gagal_24jam"
FITUR_INPUT = ["tipe_produk", "suhu_udara_K", "suhu_proses_K", "kecepatan_rpm", "torsi_Nm",
               "keausan_tool_menit", "getaran_mm_s", "jam_sejak_servis", "shift_operator"]
X = df[FITUR_INPUT]
y = df[TARGET]

In [ ]:
# TODO 5: bagi jadi 60% train, 20% validation, 20% test, semuanya stratified.
# Petunjuk: pakai train_test_split dua kali.
#   1) pisahkan 60% train dari 40% sisanya (test_size=0.4, stratify=y)
#      -> hasilnya X_train, X_sisa, y_train, y_sisa
#   2) bagi 40% sisa itu jadi dua sama besar (test_size=0.5, stratify=y_sisa)
#      -> hasilnya X_val, X_test, y_val, y_test
# Jangan lupa random_state=RANDOM_STATE supaya hasilnya sama dengan teman-teman.
X_train, X_sisa, y_train, y_sisa = ...
X_val, X_test, y_val, y_test = ...

In [ ]:
# ✅ Cek TODO 5
assert (len(X_train), len(X_val), len(X_test)) == (6000, 2000, 2000), \
    f"Ukuran split: {len(X_train)}, {len(X_val)}, {len(X_test)}. Harusnya 6000, 2000, 2000."
for nama, bagian in [("train", y_train), ("val", y_val), ("test", y_test)]:
    assert abs(bagian.mean() - y.mean()) < 0.003, f"Proporsi gagal di {nama} melenceng. Sudah pakai stratify?"
print("✅ TODO 5 beres:", {n: f"{b.mean():.1%}" for n, b in [("train", y_train), ("val", y_val), ("test", y_test)]})

## 6 · Pipeline dan model pertama

Sampai sini langkahnya sudah banyak: tambah fitur, isi nilai kosong, ubah teks jadi angka, baru masuk model.
Kalau langkah-langkah ini dijalankan terpisah, gampang ada yang kelupaan waktu model dipakai.

**Pipeline** membungkus semuanya jadi satu objek:

`data mentah → tambah_fitur → isi nilai kosong + ubah teks jadi angka → model → peluang rusak`

- Nilai kosong diisi **median**, dan median itu cuma dihitung dari data train. Kalau dihitung dari semua data,
  model sudah "ngintip" sedikit informasi dari data test. Ini yang disebut *data leakage*.
- Kolom teks (`tipe_produk`, `shift_operator`) diubah jadi angka pakai **one-hot**: tiap kategori jadi
  satu kolom berisi 0 atau 1.
- Nanti cukup panggil `pipeline.predict_proba(data_mentah)`, semua langkah jalan otomatis.

Model pertama kita **Random Forest**: banyak pohon keputusan dilatih terpisah, lalu hasilnya di-voting.
Ini jadi patokan awal. Sel ini sudah jadi, tinggal jalankan.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

KOLOM_ANGKA = ["suhu_udara_K", "suhu_proses_K", "kecepatan_rpm", "torsi_Nm", "keausan_tool_menit",
               "getaran_mm_s", "jam_sejak_servis", "daya_W", "delta_suhu_K", "strain_tool"]
KOLOM_KATEGORI = ["tipe_produk", "shift_operator"]


def buat_pipeline(model):
    preprocess = ColumnTransformer([
        ("angka", SimpleImputer(strategy="median"), KOLOM_ANGKA),
        ("kategori", OneHotEncoder(handle_unknown="ignore"), KOLOM_KATEGORI),
    ])
    return Pipeline([
        ("fitur", FunctionTransformer(tambah_fitur)),
        ("preprocess", preprocess),
        ("model", model),
    ])


rf = buat_pipeline(RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced",
                                          random_state=RANDOM_STATE, n_jobs=-1))
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_val)[:, 1]

print(f"Akurasi Random Forest      : {accuracy_score(y_val, proba_rf >= 0.5):.1%}")
print(f"Akurasi 'selalu bilang aman': {1 - y_val.mean():.1%}   <- hampir sama! akurasi menipu")
print(f"PR-AUC Random Forest (val)  : {average_precision_score(y_val, proba_rf):.3f}")
print(f"PR-AUC tebak acak           : {y_val.mean():.3f}")

Akurasi Random Forest hampir sama dengan model yang selalu bilang "aman". Artinya akurasi nggak bisa dipakai
di sini, karena mesin yang rusak cuma sedikit. Kita pakai dua ukuran lain:

- **Recall**: dari semua mesin yang beneran rusak, berapa persen yang berhasil ditangkap model.
- **PR-AUC**: angka 0 sampai 1, seberapa bagus model menangkap mesin rusak tanpa kebanyakan alarm palsu.
  Tebak acak dapat sekitar 0,055. Makin dekat ke 1 makin bagus.

## 7 · Model kedua: XGBoost

Random Forest melatih banyak pohon **terpisah** lalu voting. XGBoost (gradient boosting) membuat pohon
**satu per satu**: pohon kedua fokus memperbaiki yang salah ditebak pohon pertama, pohon ketiga memperbaiki
sisanya, dan seterusnya.

Mesin rusak jumlahnya jauh lebih sedikit: tiap 1 mesin rusak ada sekitar 17 mesin aman. Kalau nggak dikasih tahu,
model cenderung ikut mayoritas dan bilang "aman" terus. `scale_pos_weight` memberi tahu XGBoost bahwa
satu mesin rusak bobotnya kira-kira 17 kali mesin aman.

### TODO 6

In [ ]:
from xgboost import XGBClassifier

# TODO 6a: hitung scale_pos_weight = jumlah kelas 0 / jumlah kelas 1 di y_train
scale_pos_weight = ...

# TODO 6b: pipeline XGBoost sudah dibuat di bawah. Tinggal fit ke data train.
xgb = buat_pipeline(XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE, n_jobs=-1))
...

proba_xgb = xgb.predict_proba(X_val)[:, 1]

In [ ]:
# ✅ Cek TODO 6
assert 15 < scale_pos_weight < 20, f"scale_pos_weight = {scale_pos_weight}. Harusnya sekitar 17."
pr_auc_rf = average_precision_score(y_val, proba_rf)
pr_auc_xgb = average_precision_score(y_val, proba_xgb)
print(f"✅ TODO 6 beres. scale_pos_weight = {scale_pos_weight:.1f}")
print(f"PR-AUC validation  ·  Random Forest {pr_auc_rf:.3f}  vs  XGBoost {pr_auc_xgb:.3f}")

XGBoost menang, tapi tipis. Ini sering kejadian: ganti algoritma biasanya cuma menambah sedikit.
Kalau mau naik banyak, biasanya dari data yang lebih bersih atau fitur yang lebih bagus.
Kita lanjut pakai XGBoost karena kecil, cepat, dan gampang di-tuning.

### Fitur mana yang paling dipakai model?
Grafik di bawah menunjukkan seberapa besar tiap fitur dipakai XGBoost untuk menebak.
Cek: fitur fisika buatan kalian di TODO 4 ada di urutan berapa?

Fitur yang nilainya hampir nol bisa dipertimbangkan untuk dibuang (*feature selection*).
Dengan cuma belasan fitur seperti sekarang, belum perlu.

In [ ]:
nama_fitur = [n.split("__")[1] for n in xgb.named_steps["preprocess"].get_feature_names_out()]
penting = pd.Series(xgb.named_steps["model"].feature_importances_, index=nama_fitur).sort_values()
fisika = {"daya_W", "delta_suhu_K", "strain_tool"}

plt.figure(figsize=(7, 4.5))
penting.plot.barh(color=["tab:orange" if n in fisika else "tab:blue" for n in penting.index])
plt.title("Fitur yang paling dipakai XGBoost (oranye = fitur fisika TODO 4)")
plt.xlabel("importance")
plt.tight_layout()
plt.show()

## 8 · Pilih threshold pakai rupiah, bukan 0,5

Model nggak langsung bilang "rusak" atau "aman". Dia kasih **peluang** antara 0 sampai 1.
Kita yang menentukan mulai angka berapa dianggap "segera servis". Angka batas itu namanya **threshold**.
Biasanya orang pakai 0,5. Tapi coba lihat dari sisi pabrik, ada dua jenis salah:

| Salahnya | Kejadiannya | Biaya |
|---|---|---|
| **False negative** | mesin mau rusak, model bilang aman, mesinnya rusak beneran | **Rp 15 juta** |
| **False positive** | mesin sehat, model bilang servis, teknisi ngecek percuma | Rp 500 ribu |

Salah yang pertama 30 kali lebih mahal. Jadi wajar kalau model kita dibuat lebih gampang curiga.

### TODO 7 · cari threshold yang total biayanya paling murah
Untuk tiap kandidat threshold, hitung total biayanya di data validation, lalu pilih yang paling kecil.

In [ ]:
BIAYA_FN = 15_000_000
BIAYA_FP = 500_000


def total_biaya(y_benar, proba, threshold):
    tn, fp, fn, tp = confusion_matrix(y_benar, proba >= threshold).ravel()
    return fn * BIAYA_FN + fp * BIAYA_FP


print(f"Biaya kalau pakai threshold 0.50: Rp {total_biaya(y_val, proba_xgb, 0.5) / 1e6:,.1f} jt")

In [ ]:
# TODO 7: coba semua threshold 0.01, 0.02, ..., 0.95 dan simpan yang biayanya paling kecil.
kandidat = np.round(np.arange(0.01, 0.96, 0.01), 2)
biaya = ...              # list: total_biaya(y_val, proba_xgb, t) untuk tiap t di kandidat
threshold_terbaik = ...  # threshold dengan biaya paling kecil. Petunjuk: np.argmin

In [ ]:
# ✅ Cek TODO 7
assert len(biaya) == len(kandidat), "biaya harus berisi satu angka untuk tiap threshold."
assert threshold_terbaik in kandidat and biaya[list(kandidat).index(threshold_terbaik)] == min(biaya), \
    "threshold_terbaik bukan yang biayanya paling kecil."
assert threshold_terbaik < 0.5, "Threshold termurah harusnya di bawah 0.5 (kecolongan itu mahal)."
print(f"✅ TODO 7 beres. Threshold {threshold_terbaik:.2f} → Rp {min(biaya) / 1e6:,.1f} jt "
      f"(vs Rp {total_biaya(y_val, proba_xgb, 0.5) / 1e6:,.1f} jt di 0.50)")

plt.figure(figsize=(7, 3))
plt.plot(kandidat, np.array(biaya) / 1e6)
plt.axvline(threshold_terbaik, color="tab:red", linestyle="--", label=f"termurah = {threshold_terbaik:.2f}")
plt.xlabel("threshold"); plt.ylabel("total biaya (juta Rp)"); plt.legend(); plt.tight_layout(); plt.show()

### 🔓 Buka test set, sekali saja

Model dan threshold sudah dipilih pakai data validation. Sekarang baru kita lihat hasilnya di "ujian beneran".
Setelah lihat angka ini, jangan balik ke atas buat ngubah-ngubah model lagi.

In [ ]:
proba_test = xgb.predict_proba(X_test)[:, 1]
pr_auc_test = average_precision_score(y_test, proba_test)
recall_test = recall_score(y_test, proba_test >= threshold_terbaik)
tn, fp, fn, tp = confusion_matrix(y_test, proba_test >= threshold_terbaik).ravel()

print(f"PR-AUC test          : {pr_auc_test:.3f}")
print(f"Recall kelas gagal   : {recall_test:.2f}  ({tp} dari {tp + fn} mesin gagal ketangkep)")
print(f"Alarm palsu          : {fp} dari {tn + fp} mesin sehat")
print(f"Total biaya test     : Rp {total_biaya(y_test, proba_test, threshold_terbaik) / 1e6:,.1f} jt")

## 9 · Simpan model beserta catatannya

Model kita masih ada di memori Colab. Kalau Colab ditutup, hilang. Jadi kita simpan ke file:

- `artifacts/model.joblib`: seluruh pipeline (tambah fitur, isi nilai kosong, one-hot, model).
- `artifacts/metadata.json`: catatan tentang modelnya: threshold, daftar kolom input, hasil di test set,
  asumsi biaya, dan versi library. Tanpa catatan ini, tiga bulan lagi nggak ada yang tahu threshold-nya
  berapa atau inputnya apa. Kayak obat tanpa label.
- `src/features.py`: sudah ditulis di TODO 4. `model.joblib` butuh file ini, jadi selalu dibawa bareng.

### TODO 8

In [ ]:
import joblib
import sklearn, xgboost

# TODO 8a: simpan pipeline xgb ke artifacts/model.joblib. Petunjuk: joblib.dump(objek, path)
...

# TODO 8b: lengkapi dua isian di metadata, lalu jalankan
metadata = {
    "versi_model": "1.0.0",
    "dibuat": time.strftime("%Y-%m-%d"),
    "fitur_input": ...,        # daftar 9 kolom input (variabelnya sudah ada di bagian 5)
    "target": TARGET,
    "threshold": ...,          # threshold hasil TODO 7
    "metrik_validasi": {"pr_auc": round(float(pr_auc_xgb), 3)},
    "metrik_test": {"pr_auc": round(float(pr_auc_test), 3), "recall": round(float(recall_test), 3)},
    "asumsi_biaya": {"FN": BIAYA_FN, "FP": BIAYA_FP},
    "versi_library": {"python": sys.version.split()[0], "sklearn": sklearn.__version__,
                      "xgboost": xgboost.__version__, "pandas": pd.__version__},
}
with open("artifacts/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

In [ ]:
# ✅ Cek TODO 8
assert os.path.exists("artifacts/model.joblib"), "artifacts/model.joblib belum ada."
meta = json.load(open("artifacts/metadata.json"))
assert meta["fitur_input"] == FITUR_INPUT, "fitur_input harus berisi 9 kolom input, urutannya sama."
assert meta["threshold"] == threshold_terbaik, "threshold di metadata harus hasil TODO 7."
ukuran = os.path.getsize("artifacts/model.joblib") / 1e6
print(f"✅ TODO 8 beres. model.joblib {ukuran:.1f} MB")
print(json.dumps(meta, indent=2))

## 10 · Coba pakai model dari file

Tes terakhir. Pura-pura notebook-nya sudah ditutup: kita **muat ulang** model dari file, bukan pakai variabel
`xgb` yang masih ada di memori. Lalu kita tebak satu mesin.

Ini sering kelupaan. Banyak model yang jalan di notebook, tapi pas dibuka dari file ternyata error.

In [ ]:
model = joblib.load("artifacts/model.joblib")
meta = json.load(open("artifacts/metadata.json"))

request = {
    "tipe_produk": "L",
    "suhu_udara_K": 301.2,
    "suhu_proses_K": 310.8,
    "kecepatan_rpm": 1290,
    "torsi_Nm": 62.5,
    "keausan_tool_menit": 228,
    "getaran_mm_s": 4.1,
    "jam_sejak_servis": 410,
    "shift_operator": "malam",
}

mulai = time.perf_counter()
proba = float(model.predict_proba(pd.DataFrame([request])[meta["fitur_input"]])[0, 1])
response = {
    "probabilitas_gagal": round(proba, 4),
    "status": "SEGERA SERVIS" if proba >= meta["threshold"] else "AMAN",
    "threshold": meta["threshold"],
    "versi_model": meta["versi_model"],
    "latency_ms": round((time.perf_counter() - mulai) * 1000, 1),
}
print(json.dumps(response, indent=2))

Masuk akal? Tool-nya sudah aus (228 menit, lewat batas 200 menit tadi) dan torsinya tinggi (62,5 Nm).
Persis pola yang kita lihat di grafik bagian 2. Coba ubah `keausan_tool_menit` jadi 10 dan `torsi_Nm` jadi 35, lalu jalankan ulang.

Sel berikutnya mengecek kasus sensor rusak: torsi dan getaran kosong. Model tetap harus bisa menjawab.

In [ ]:
# Sensor rusak juga harus bisa ditangani: torsi kosong dan getaran kosong
request_rusak = {**request, "torsi_Nm": None, "getaran_mm_s": None}
proba_rusak = model.predict_proba(pd.DataFrame([request_rusak])[meta["fitur_input"]])[0, 1]
print(f"Request dengan sensor kosong tetap dapat jawaban: {proba_rusak:.4f}")

### Bawa ke P3

Sel di bawah membungkus tiga file jadi satu zip dan mengunduhnya otomatis. **Simpan baik-baik**, dibawa ke P3:

`artifacts/model.joblib` · `artifacts/metadata.json` · `src/features.py`

In [ ]:
import zipfile

with zipfile.ZipFile("artifact_p2.zip", "w") as z:
    for path in ["artifacts/model.joblib", "artifacts/metadata.json", "src/features.py"]:
        z.write(path)
print("artifact_p2.zip siap:", os.path.getsize("artifact_p2.zip") // 1024, "KB")

try:
    from google.colab import files
    files.download("artifact_p2.zip")
except ImportError:
    print("Bukan di Colab: ambil artifact_p2.zip dari folder kerja.")

## PR sebelum P3

Pilih **satu** aja. Kerjain di salinan notebook kalian sendiri, terus catat angkanya buat dibahas di awal P3.

**1. Coba algoritma lain: LightGBM.**
Bikin sel baru di bawah bagian 7, isi dengan:

```python
from lightgbm import LGBMClassifier

lgbm = buat_pipeline(LGBMClassifier(
    n_estimators=300, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE, verbose=-1))
lgbm.fit(X_train, y_train)
print("PR-AUC LightGBM:", average_precision_score(y_val, lgbm.predict_proba(X_val)[:, 1]))
```

Pertanyaannya: PR-AUC validation-nya lebih tinggi dari XGBoost (0,649) atau nggak?

**2. Bikin satu fitur baru.**
Di sel `tambah_fitur` (bagian 4), tambah satu baris kolom baru dari kolom yang sudah ada. Contoh:
`df["jarak_putar"] = df["keausan_tool_menit"] * df["kecepatan_rpm"]` (berapa banyak tool sudah berputar).
Lalu tambahkan nama kolomnya ke `KOLOM_ANGKA` di bagian 6, dan jalankan ulang dari bagian 4 ke bawah.

Pertanyaannya: PR-AUC validation XGBoost naik, turun, atau sama aja? Kenapa menurut kalian?

**3. Ubah asumsi biaya.**
Di bagian 8, ganti `BIAYA_FN` (biaya mesin rusak yang kelewat) dari Rp 15 juta jadi **Rp 5 juta**, lalu jalankan
ulang sel itu dan sel TODO 7. Catat threshold barunya. Habis itu coba **Rp 50 juta**.

Pertanyaannya: threshold-nya geser ke mana di masing-masing kasus, dan kenapa? Kalau ada yang nggak geser,
coba cari tahu sebabnya. Petunjuk: hitung berapa mesin rusak yang tetap kelewat di threshold itu.

Mentok? Tanya di QnA dan tempel pesan error lengkapnya.